# CycleGAN — Unpaired Image-to-Image Translation

This notebook implements **CycleGAN** (Zhu et al., 2017), following Module 16.

Pix2Pix solved paired image translation, but real applications rarely come with `(x, y)` correspondences. CycleGAN drops that requirement and learns from **two collections** of images with no explicit pairing.

The conceptual jump is big:

```
Pix2Pix:   G(x) ~ y,           given paired (x, y)
CycleGAN:  G(x_A) ~ domain B,  given two collections A and B (unpaired)
           F(x_B) ~ domain A
           F(G(x_A)) ~ x_A     (cycle consistency)
           G(F(x_B)) ~ x_B
```

Three losses do the work:

- **Adversarial** for each direction — make `G(A)` look like real `B` and `F(B)` look like real `A`.
- **Cycle-consistency** — applying `G` then `F` should recover `x_A`, and applying `F` then `G` should recover `x_B`. Without this, the Generators can produce realistic-but-arbitrary outputs.
- **Identity** (optional) — feeding an image from domain `B` into `G` (which is supposed to map `A -> B`) should leave it mostly alone. This is a color/composition regularizer.

**Our unpaired task.** Two MNIST collections: domain `A` is the standard digit, domain `B` is the same digit rotated 180 degrees. Each domain comes from MNIST, but the model never sees the pairing — it just sees two streams of images.

## Implementation Plan

- **Two Generators** `G: A -> B` and `F: B -> A`. ResNet-style: 7x7 initial conv with reflection padding, two strided convs for downsampling to 16x16, six residual blocks at 16x16, two transposed convs for upsampling back to 64x64, 7x7 final conv with `Tanh`. Uses **InstanceNorm** (the paper's convention for CycleGAN).
- **Two Discriminators** `D_A` and `D_B`. PatchGAN — same 70x70 receptive-field idea from Pix2Pix, output an 8x8 logit map.
- **Three losses per Generator step**: adversarial (`BCEWithLogitsLoss` against target `1`), cycle consistency (`L1Loss` between reconstructions and inputs scaled by `lambda_cyc`), and identity (`L1Loss` between `G(y)` and `y` scaled by `lambda_id`).
- **Two Discriminator steps**: each `D` is trained on its real-domain batch and on the *detached* fake from the corresponding Generator.
- **Two MNIST loaders**: one produces domain A (no transform), the other produces domain B (180 degree rotation). Same underlying dataset, but the model sees two independent streams and never the pairing.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.utils import save_image, make_grid
import matplotlib.pyplot as plt
import numpy as np
import os

torch.manual_seed(42)
np.random.seed(42)

## 1. Setup and Hyperparameters

CycleGAN has four networks to train, so we drop `batch_size` to keep memory reasonable. The two loss weights control the trade-off:

- `lambda_cyc = 10` (paper default) — the cycle-consistency term.
- `lambda_id  = 5` (paper uses 0.5 * lambda_cyc) — the identity regularizer.

In [ ]:
img_channels = 1
img_size     = 64
features_g   = 64
features_d   = 64
n_resblocks  = 6
batch_size   = 16
lr           = 2e-4
betas        = (0.5, 0.999)
epochs       = 15
lambda_cyc   = 10.0
lambda_id    = 5.0

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

os.makedirs('samples_CycleGAN', exist_ok=True)

## 2. Data — Two Unpaired MNIST Streams

Domain A: standard MNIST digits.
Domain B: the same MNIST digits, rotated 180 degrees.

We use **two separate DataLoaders**. From the model's perspective these are unpaired collections. The pairing exists in the data preparation but is never shown to the network.

In [ ]:
transform_A = transforms.Compose([
    transforms.Resize(img_size),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

transform_B = transforms.Compose([
    transforms.Resize(img_size),
    transforms.RandomRotation((180, 180)),    # domain B: rotated 180 deg
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

dataset_A = datasets.MNIST('./data', train=True, download=True, transform=transform_A)
dataset_B = datasets.MNIST('./data', train=True, download=True, transform=transform_B)

loader_A = torch.utils.data.DataLoader(dataset_A, batch_size=batch_size, shuffle=True, drop_last=True)
loader_B = torch.utils.data.DataLoader(dataset_B, batch_size=batch_size, shuffle=True, drop_last=True)

## 3. Weight Initialization

DCGAN-style init, with `InstanceNorm` handled by giving it weight=1, bias=0 in the same spirit.

In [ ]:
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('InstanceNorm') != -1:
        if m.weight is not None:
            nn.init.normal_(m.weight.data, 1.0, 0.02)
        if m.bias is not None:
            nn.init.constant_(m.bias.data, 0)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)
    elif classname.find('Linear') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)

## 4. The Generator — ResNet with Reflection Padding

Spatial flow for a 64x64 input:

```
initial conv (7x7, reflect pad) -> 64x64,  f
downsample (stride 2)            -> 32x32,  2f
downsample (stride 2)            -> 16x16,  4f
[6 residual blocks]              -> 16x16,  4f
upsample (stride 2)              -> 32x32,  2f
upsample (stride 2)              -> 64x64,  f
final conv (7x7, reflect pad) + Tanh -> 64x64,  img_channels
```

Two non-obvious details:

1. **Reflection padding** (instead of zero padding) before each 7x7 conv. The paper found that reflection padding avoids the checkerboard / boundary artifacts that zero padding produces in image-to-image networks.
2. **InstanceNorm** instead of BatchNorm. CycleGAN's authors found that InstanceNorm works much better than BatchNorm in this setting — each image gets its own mean/std normalization, which is appropriate for style translation.

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, 3, bias=False),
            nn.InstanceNorm2d(channels),
            nn.ReLU(True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, 3, bias=False),
            nn.InstanceNorm2d(channels),
        )

    def forward(self, x):
        return x + self.block(x)


class Generator(nn.Module):
    def __init__(self, img_channels=1, features=64, n_resblocks=6):
        super().__init__()

        # Initial conv: 64x64, features
        self.initial = nn.Sequential(
            nn.ReflectionPad2d(3),
            nn.Conv2d(img_channels, features, 7, bias=False),
            nn.InstanceNorm2d(features),
            nn.ReLU(True),
        )

        # Downsample: 64 -> 32 -> 16, doubling channels each step
        self.down = nn.Sequential(
            nn.Conv2d(features,     features * 2, 3, 2, 1, bias=False),
            nn.InstanceNorm2d(features * 2),
            nn.ReLU(True),
            nn.Conv2d(features * 2, features * 4, 3, 2, 1, bias=False),
            nn.InstanceNorm2d(features * 4),
            nn.ReLU(True),
        )

        # Residual blocks at 16x16
        self.res = nn.Sequential(
            *[ResBlock(features * 4) for _ in range(n_resblocks)]
        )

        # Upsample: 16 -> 32 -> 64, halving channels each step
        self.up = nn.Sequential(
            nn.ConvTranspose2d(features * 4, features * 2, 3, 2, 1, 1, bias=False),
            nn.InstanceNorm2d(features * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(features * 2, features,     3, 2, 1, 1, bias=False),
            nn.InstanceNorm2d(features),
            nn.ReLU(True),
        )

        # Final conv back to img_channels + Tanh
        self.final = nn.Sequential(
            nn.ReflectionPad2d(3),
            nn.Conv2d(features, img_channels, 7),
            nn.Tanh(),
        )

    def forward(self, x):
        x = self.initial(x)        # 64x64, f
        x = self.down(x)           # 16x16, 4f
        x = self.res(x)            # 16x16, 4f
        x = self.up(x)             # 64x64, f
        return self.final(x)

## 5. The Discriminator — PatchGAN

Same architecture as Pix2Pix's PatchGAN, but now there are *two* of them — one per domain. Each Discriminator sees only its own domain and judges local patch realism.

In [ ]:
class PatchDiscriminator(nn.Module):
    def __init__(self, img_channels=1, features=64):
        super().__init__()
        self.net = nn.Sequential(
            # 64 -> 32, NO normalization in the first block
            nn.Conv2d(img_channels, features, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            # 32 -> 16
            nn.Conv2d(features, features * 2, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(features * 2),
            nn.LeakyReLU(0.2, inplace=True),

            # 16 -> 8
            nn.Conv2d(features * 2, features * 4, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(features * 4),
            nn.LeakyReLU(0.2, inplace=True),

            # 8 -> 8 (stride 1, kernel 3)
            nn.Conv2d(features * 4, features * 8, 3, 1, 1, bias=False),
            nn.InstanceNorm2d(features * 8),
            nn.LeakyReLU(0.2, inplace=True),

            # final 1-channel logit map (8x8)
            nn.Conv2d(features * 8, 1, 3, 1, 1, bias=False),
        )

    def forward(self, x):
        return self.net(x)

## 6. Models, Optimizers, Losses

Four networks. Two optimizers (one for the two Generators, one for the two Discriminators). Same `BCEWithLogitsLoss` for adversarial, plus `L1Loss` for cycle and identity.

In [ ]:
G = Generator(img_channels=img_channels, features=features_g, n_resblocks=n_resblocks).to(device)  # A -> B
F = Generator(img_channels=img_channels, features=features_g, n_resblocks=n_resblocks).to(device)  # B -> A
D_A = PatchDiscriminator(img_channels=img_channels, features=features_d).to(device)
D_B = PatchDiscriminator(img_channels=img_channels, features=features_d).to(device)

G.apply(weights_init)
F.apply(weights_init)
D_A.apply(weights_init)
D_B.apply(weights_init)

optimizer_G = optim.Adam(list(G.parameters()) + list(F.parameters()), lr=lr, betas=betas)
optimizer_D = optim.Adam(list(D_A.parameters()) + list(D_B.parameters()), lr=lr, betas=betas)

bce = nn.BCEWithLogitsLoss()
l1  = nn.L1Loss()

print(f'G params: {sum(p.numel() for p in G.parameters()):,}')
print(f'F params: {sum(p.numel() for p in F.parameters()):,}')
print(f'D_A params: {sum(p.numel() for p in D_A.parameters()):,}')
print(f'D_B params: {sum(p.numel() for p in D_B.parameters()):,}')

## 7. Fixed Test Examples for Visualization

Sample a fixed batch from each domain. We'll run these through both Generators every epoch to visualize how `G(A) -> B`, `F(B) -> A`, and the cycle reconstructions evolve.

In [ ]:
fixed_A = next(iter(loader_A))[0].to(device)[:6]
fixed_B = next(iter(loader_B))[0].to(device)[:6]
print('fixed_A:', tuple(fixed_A.shape))
print('fixed_B:', tuple(fixed_B.shape))

## 8. The Training Loop

Three steps per iteration, updating four networks.

**Step 1 — Train `D_A`**

- Real A from `loader_A` -> `D_A` -> target 1
- Fake A = `F(real_B).detach()` -> `D_A` -> target 0
- BCE sum, backprop into D_A.

**Step 2 — Train `D_B`**

- Real B from `loader_B` -> `D_B` -> target 1
- Fake B = `G(real_A).detach()` -> `D_B` -> target 0
- BCE sum, backprop into D_B.

**Step 3 — Train G and F together**

1. `fake_B = G(real_A)` and `fake_A = F(real_B)` (no detach).
2. `recon_A = F(fake_B)` and `recon_B = G(fake_A)` — the round trips.
3. `id_B = G(real_B)` and `id_A = F(real_A)` — identity preservation.
4. **Adversarial**: `BCE(D_B(fake_B), 1)` + `BCE(D_A(fake_A), 1)`.
5. **Cycle**: `L1(recon_A, real_A) + L1(recon_B, real_B)` scaled by `lambda_cyc`.
6. **Identity**: `L1(id_B, real_B) + L1(id_A, real_A)` scaled by `lambda_id`.
7. Sum, backprop through both Generators.

The cycle loss is what keeps the Generators honest. Without it, `G` could map every A-image to the same plausible B-image and `D_B` would be happy.

In [ ]:
losses_d     = []
losses_g     = []
losses_g_adv = []
losses_g_cyc = []
losses_g_id  = []

iter_A = iter(loader_A)
iter_B = iter(loader_B)

n_batches = min(len(loader_A), len(loader_B))

G.train(); F.train(); D_A.train(); D_B.train()

for epoch in range(epochs):
    sum_d     = 0.0
    sum_g     = 0.0
    sum_adv   = 0.0
    sum_cyc   = 0.0
    sum_id    = 0.0

    for _ in range(n_batches):
        try:
            real_A = next(iter_A)[0].to(device)
        except StopIteration:
            iter_A = iter(loader_A)
            real_A = next(iter_A)[0].to(device)
        try:
            real_B = next(iter_B)[0].to(device)
        except StopIteration:
            iter_B = iter(loader_B)
            real_B = next(iter_B)[0].to(device)

        b = real_A.size(0)
        valid = torch.ones (b, 1, 8, 8, device=device)
        fake_t = torch.zeros(b, 1, 8, 8, device=device)

        # -----------------
        # Step 1: Train D_A
        # -----------------
        optimizer_D.zero_grad()
        d_A_real = bce(D_A(real_A), valid)
        with torch.no_grad():
            fake_A = F(real_B)
        d_A_fake = bce(D_A(fake_A), fake_t)
        d_A_loss = (d_A_real + d_A_fake) / 2

        # -----------------
        # Step 2: Train D_B
        # -----------------
        d_B_real = bce(D_B(real_B), valid)
        with torch.no_grad():
            fake_B = G(real_A)
        d_B_fake = bce(D_B(fake_B), fake_t)
        d_B_loss = (d_B_real + d_B_fake) / 2

        d_loss = d_A_loss + d_B_loss
        d_loss.backward()
        optimizer_D.step()

        # -----------------
        # Step 3: Train G and F
        # -----------------
        optimizer_G.zero_grad()

        # Forward cycle: A -> B -> A
        fake_B = G(real_A)
        recon_A = F(fake_B)

        # Backward cycle: B -> A -> B
        fake_A = F(real_B)
        recon_B = G(fake_A)

        # Identity
        id_B = G(real_B)
        id_A = F(real_A)

        # Adversarial
        g_adv = bce(D_B(fake_B), valid) + bce(D_A(fake_A), valid)

        # Cycle
        g_cyc = (l1(recon_A, real_A) + l1(recon_B, real_B)) * lambda_cyc

        # Identity
        g_id = (l1(id_B, real_B) + l1(id_A, real_A)) * lambda_id

        g_loss = g_adv + g_cyc + g_id
        g_loss.backward()
        optimizer_G.step()

        sum_d   += d_loss.item()
        sum_g   += g_loss.item()
        sum_adv += g_adv.item()
        sum_cyc += g_cyc.item()
        sum_id  += g_id.item()

    avg_d   = sum_d   / n_batches
    avg_g   = sum_g   / n_batches
    avg_adv = sum_adv / n_batches
    avg_cyc = sum_cyc / n_batches
    avg_id  = sum_id  / n_batches
    losses_d.append(avg_d)
    losses_g.append(avg_g)
    losses_g_adv.append(avg_adv)
    losses_g_cyc.append(avg_cyc)
    losses_g_id.append(avg_id)

    print(
        f"Epoch [{epoch+1}/{epochs}]  "
        f"D: {avg_d:.3f}  "
        f"G: {avg_g:.3f}  (adv {avg_adv:.3f}, cyc {avg_cyc:.3f}, id {avg_id:.3f})"
    )

    # Save a 6-row comparison grid every epoch:
    #   row 1: real A
    #   row 2: G(A)  -> B
    #   row 3: F(G(A)) -> reconstruction of A
    #   row 4: real B
    #   row 5: F(B)  -> A
    #   row 6: G(F(B)) -> reconstruction of B
    G.eval(); F.eval()
    with torch.no_grad():
        fake_B_fixed  = G(fixed_A)
        recon_A_fixed = F(fake_B_fixed)
        fake_A_fixed  = F(fixed_B)
        recon_B_fixed = G(fake_A_fixed)
    G.train(); F.train()

    comparison = torch.cat([
        fixed_A.cpu(),
        fake_B_fixed.cpu(),
        recon_A_fixed.cpu(),
        fixed_B.cpu(),
        fake_A_fixed.cpu(),
        recon_B_fixed.cpu(),
    ], dim=0)
    save_image(
        comparison,
        f"samples_CycleGAN/epoch_{epoch+1:02d}.png",
        nrow=6,
        normalize=True,
    )

print('Training done.')

## 9. Visualizing the Cycle

Six rows stacked vertically:

1. Real A (input)
2. `G(A)` — translation A -> B
3. `F(G(A))` — round-trip reconstruction of A (should look like row 1)
4. Real B (input)
5. `F(B)` — translation B -> A
6. `G(F(B))` — round-trip reconstruction of B (should look like row 4)

Rows 1 and 3 should match; rows 4 and 6 should match. Rows 2 and 5 are the actual translations.

In [ ]:
G.eval(); F.eval()
with torch.no_grad():
    fake_B_fixed  = G(fixed_A).cpu()
    recon_A_fixed = F(fake_B_fixed.to(device)).cpu()
    fake_A_fixed  = F(fixed_B).cpu()
    recon_B_fixed = G(fake_A_fixed.to(device)).cpu()

comparison = torch.cat([
    fixed_A.cpu(),
    fake_B_fixed,
    recon_A_fixed,
    fixed_B.cpu(),
    fake_A_fixed,
    recon_B_fixed,
], dim=0)
grid = make_grid(comparison, nrow=6, normalize=True)
plt.figure(figsize=(10, 12))
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap='gray')
plt.axis('off')
plt.title(
    "CycleGAN — row 1: real A | row 2: G(A) | row 3: F(G(A)) | "
    "row 4: real B | row 5: F(B) | row 6: G(F(B))"
)
plt.show()

## 10. Training Progression

Each saved PNG is the full six-row stack for the fixed test batch. Rows 1 and 3 should converge to look the same; rows 4 and 6 likewise.

In [ ]:
from PIL import Image
import glob

paths = sorted(glob.glob('samples_CycleGAN/epoch_*.png'))
if paths:
    fig, axes = plt.subplots(1, len(paths), figsize=(2.2 * len(paths), 2.2))
    if len(paths) == 1:
        axes = [axes]
    for ax, p in zip(axes, paths):
        ax.imshow(np.array(Image.open(p)).squeeze(), cmap='gray')
        ax.set_title(p.split('_')[-1].split('.')[0])
        ax.axis('off')
    plt.suptitle('CycleGAN progression — 6-row cycle visualization')
    plt.show()
else:
    print('No sample grids found. Run the training cell first.')

## 11. Loss Curves

The Generator-side split shows three signals:

- **G-adv** — adversarial term, hovers around a small positive number.
- **G-cyc** — cycle term, scaled by `lambda_cyc = 10`. Should fall as the round-trip becomes more faithful.
- **G-id** — identity term, scaled by `lambda_id = 5`. Tracks color/composition preservation.

On MNIST these are noisy in absolute terms (small dataset, sharp domain shift), but the cycle curve falling is the strongest single signal that the model is learning.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(losses_d, label='D total', color='tab:blue')
axes[0].set_xlabel('Epoch')
axes[0].set_title('Discriminator loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(losses_g,     label='G total',     color='tab:orange')
axes[1].plot(losses_g_adv, label='G adv',       color='tab:red',    linestyle='--')
axes[1].plot(losses_g_cyc, label='G cyc * 10',  color='tab:green',  linestyle='--')
axes[1].plot(losses_g_id,  label='G id * 5',    color='tab:purple', linestyle='--')
axes[1].set_xlabel('Epoch')
axes[1].set_title('Generator loss (cycle and identity are scaled)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Recap — How CycleGAN Differs From Pix2Pix

| Concern | Pix2Pix | CycleGAN |
|---|---|---|
| Data | **paired** `(x, y)` | **unpaired** collections `A`, `B` |
| Generators | 1 | **2** — `G: A->B`, `F: B->A` |
| Discriminators | 1 (PatchGAN on `(x, y)` pair) | **2** — `D_A`, `D_B` |
| Reconstruction loss | L1 to paired target | **cycle L1** to original input |
| Identity loss | none | **optional** — color/composition regularizer |
| Distribution modeled | `p(y | x)` for paired data | `p_B` reachable from `A`, `p_A` reachable from `B` |
| Generator architecture | U-Net | **ResNet** with residual blocks |
| Normalization | BatchNorm | **InstanceNorm** |
| Padding | zero | **reflection** |

**What changed and why.**

- **Two Generators** so that the round trip `A -> B -> A` is well-defined. Without `F`, there is no `recon_A` to compare against the original input, and the cycle loss collapses to nothing.
- **Two Discriminators** because each domain has its own realism criterion. `D_A` doesn't need to know anything about what `B` looks like — it only judges whether something looks like `A`.
- **Cycle consistency** is the entire point. Adversarial loss alone is satisfied by any plausible B-image; cycle loss forces the round trip to be information-preserving. The two together give you *style translation* rather than *arbitrary stylization*.
- **InstanceNorm over BatchNorm, reflection padding over zero padding.** CycleGAN's authors found both choices matter for image-to-image translation. InstanceNorm normalizes per-image, which fits style transfer better; reflection padding avoids the boundary artifacts that zero padding produces in deep conv stacks.
- **ResNet generator** over U-Net. Both work; ResNet is the original CycleGAN choice. Residual blocks let the network apply a transformation around the input representation, which is the right inductive bias for "change the style but keep the content."

**Common implementation pitfalls** — quick reference:

- **Identity loss is not the same as cycle loss.** Cycle loss measures the round trip. Identity loss measures what happens when you feed an image from the target domain into a Generator that shouldn't need to translate it. They have different gradient signals; cycle alone doesn't preserve color/composition the way identity does.
- **Detach the fakes during D updates.** If you forget `torch.no_grad()` (or `.detach()`) when feeding fakes to the Discriminator, the gradients sneak into the Generator. With two Generators and two Discriminators the bookkeeping is easy to get wrong.
- **Cycle loss dominates the loss scale.** With `lambda_cyc = 10`, the cycle term is much larger than the adversarial term in absolute value. Don't be alarmed by the G total being mostly cycle loss — that's expected.
- **Both loaders iterate independently.** Two DataLoaders over the same dataset with different transforms will produce different orderings each epoch. That's fine and even desirable — the model never sees the explicit pairing.
- **CycleGAN is finicky.** It is the most hyperparameter-sensitive model in this series. Training collapse, mode collapse, and oscillation are all common. The standard fix is to halve the learning rate or train for longer with smaller batches.
- **InstanceNorm, not BatchNorm.** Switching to BatchNorm is a common porting mistake. It usually hurts performance noticeably.

**Why unpaired translation matters.** Most real image collections aren't paired. You have a folder of horses and a folder of zebras, not horse-zebra pairs. Pix2Pix is elegant but practically limited. CycleGAN unlocks the much larger space of *unpaired* collections — and is the foundation for almost every practical image-to-image translation tool you'll see in the wild.

**Next in the series** (Module 17): Progressive GAN. The next jump is *resolution* — training a Generator that grows from `4x4` to `1024x1024` by adding layers during training. Stable high-resolution GANs require careful growing strategies and the right loss formulation.